# **Coleta de dados**

In [1]:
#Bases de Dados 
from pathlib import Path
import pandas as pd
#import duckdb

# Visualização
import plotly.express as px

# Modelagem 
import numpy as np 
from typing import List, Optional

In [2]:
# Configurações de exibição
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Se preferir memória: 
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [63]:
path_dim_soc_vigimed = str(Path(r"C:\Users\silma\OneDrive\Área de Trabalho\dim_soc_llt_vigimed.parquet"))
path_dim_soc_llt = str(Path(r"C:\Users\silma\Projetos\vigimed\data\03_gold\dim_soc_llt\dim_soc_llt.parquet"))

In [11]:
%sql ROLLBACK;

Running query in 'duckdb:///:memory:'

,Success


In [64]:
%%sql
DROP TABLE IF EXISTS dim_soc_vigimed;
CREATE TABLE dim_soc_vigimed AS
SELECT * FROM read_parquet(:path_dim_soc_vigimed);
 
DROP TABLE IF EXISTS dim_soc_llt;
CREATE TABLE dim_soc_llt AS
SELECT *
FROM read_parquet(:path_dim_soc_llt);
 

Running query in 'duckdb:///:memory:'

,Success


In [65]:
%%sql 
select * from dim_soc_vigimed LIMIT 2

Running query in 'duckdb:///:memory:'

,SOC_CHAVE,SOC_VALOR,HLGT,HLT,PT,LLT_CHAVE,REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual


In [66]:
%%sql 
select * from dim_soc_llt LIMIT 2

Running query in 'duckdb:///:memory:'

,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME
0,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002043,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002043,Anemia por deficiência de folato
1,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002044,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002044,Anemia por deficiência de ácido fólico


In [70]:
%%sql 
select COUNT(*) AS QT,
COUNT(DISTINCT REACAO_CHAVE) AS REACAO_CHAVE,
COUNT(DISTINCT CONCAT(LLT_NAME, PT_NAME, HLT_NAME, HLGT_NAME, SOC_NAME,PRIMARY_FLAG)) as REACAO,
COUNT(DISTINCT LLT_NAME) AS LLT,
COUNT(DISTINCT LLT_CODE) AS LLT_CODE,
 COUNT(DISTINCT PT_NAME) AS PT ,
 COUNT(DISTINCT PT_CODE) AS PT_CODE,
 COUNT(DISTINCT HLT_NAME) AS HLT ,
 COUNT(DISTINCT HLT_CODE) AS HLT_CODE,
 COUNT(DISTINCT HLGT_NAME) AS HLGT ,
 COUNT(DISTINCT HLGT_CODE) AS HLGT_CODE,
 COUNT(DISTINCT SOC_NAME) AS SOC_NAME,
 COUNT(DISTINCT SOC_CODE) AS SOC_CODE,
from dim_soc_llt
;

Running query in 'duckdb:///:memory:'

,QT,REACAO_CHAVE,REACAO,LLT,LLT_CODE,PT,PT_CODE,HLT,HLT_CODE,HLGT,HLGT_CODE,SOC_NAME,SOC_CODE
0,142785,142785,125329,80002,90471,27163,27163,1739,1739,337,337,27,27


In [33]:
%%sql 
select COUNT(*) AS QT,
COUNT(DISTINCT LLT_NAME) AS LLT,
COUNT(DISTINCT LLT_CODE) AS LLT_CODE,
 COUNT(DISTINCT PT_NAME) AS PT ,
 COUNT(DISTINCT PT_CODE) AS PT_CODE,
 COUNT(DISTINCT HLT_NAME) AS HLT ,
 COUNT(DISTINCT HLT_CODE) AS HLT_CODE,
 COUNT(DISTINCT HLGT_NAME) AS HLGT ,
 COUNT(DISTINCT HLGT_CODE) AS HLGT_CODE,
 COUNT(DISTINCT SOC_NAME) AS SOC_NAME,
 COUNT(DISTINCT SOC_CODE) AS SOC_CODE
from dim_soc_llt 
where PRIMARY_FLAG = 'Y'

Running query in 'duckdb:///:memory:'

,QT,LLT,LLT_CODE,PT,PT_CODE,HLT,HLT_CODE,HLGT,HLGT_CODE,SOC_NAME,SOC_CODE
0,90471,80002,90471,27163,27163,1566,1566,326,326,27,27


In [31]:
%%sql 
select COUNT(1) AS QT, COUNT(DISTINCT REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR) AS LLT,
 COUNT(DISTINCT PT) AS PT ,
 COUNT(DISTINCT HLT) AS HLT ,
 COUNT(DISTINCT HLGT) AS HLGT ,
 COUNT(DISTINCT SOC_VALOR) AS SOC_VALOR 
from dim_soc_vigimed LIMIT 2

Running query in 'duckdb:///:memory:'

,QT,LLT,PT,HLT,HLGT,SOC_VALOR
0,19057,18698,9232,1351,321,27


In [21]:
18698-19057

-359

In [27]:
%%sql 
SELECT COUNT(1) AS QT,COUNT(DISTINCT LLT_NAME) AS LLT,
 COUNT(DISTINCT PT_NAME) AS PT ,
 COUNT(DISTINCT HLT_NAME) AS HLT ,
 COUNT(DISTINCT HLGT_NAME) AS HLGT ,
 COUNT(DISTINCT SOC_NAME) AS SOC_NAME 
from dim_soc_llt AS a 
inner join dim_soc_vigimed AS b on a.LLT_NAME = b.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
and a.PT_NAME = b.PT 

Running query in 'duckdb:///:memory:'

,QT,LLT,PT,HLT,HLGT,SOC_NAME
0,35424,18501,9080,1551,335,27


In [39]:
%%sql
with base as (
    SELECT 
        A.LLT_NAME,
        A.LLT_CODE,
        a.PT_NAME,
        a.HLT_NAME,
        a.HLGT_NAME,
        a.SOC_NAME,
        B.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR AS "LLT",
        B.SOC_VALOR AS "SOC"
    from dim_soc_llt AS a
    inner join dim_soc_vigimed AS b 
        on a.LLT_NAME = b.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
        and a.PT_NAME = b.PT
)

select LLT_CODE, COUNT(1) AS QT
from base
group by 1
HAVING COUNT(1) > 1
ORDER BY 2 DESC
LIMIT 10

Running query in 'duckdb:///:memory:'

,LLT_CODE,QT
0,10050104,8
1,10004215,7
2,10004212,7
3,10066583,7
4,10004213,7
5,10004211,7
6,10063683,6
7,10048749,6
8,10060803,6
9,10048941,6


In [ ]:
%%sql
with base as (
    SELECT distinct 
        A.LLT_NAME,
        A.LLT_CODE,
        a.PT_NAME,
        a.HLT_NAME,
        a.HLGT_NAME,
        a.SOC_NAME,
        B.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR AS "LLT",
        B.SOC_VALOR AS "SOC"
    from dim_soc_llt AS a
    inner join dim_soc_vigimed AS b 
        on a.LLT_NAME = b.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
        and a.PT_NAME = b.PT
        and a.HLT_NAME = b.HLT
        and a.HLGT_NAME = b.HLGT
        and a.SOC_NAME = b.SOC_VALOR
    where a.PRIMARY_FLAG = 'Y'
)

select COUNT(1)
from base

Running query in 'duckdb:///:memory:'

,count(1)
0,22505


In [54]:
%%sql
with base as (
    SELECT distinct 
        A.LLT_NAME,
        A.LLT_CODE,
        a.PT_NAME,
        a.HLT_NAME,
        a.HLGT_NAME,
        a.SOC_NAME,
        B.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR AS "LLT",
        B.SOC_VALOR AS "SOC"
    from dim_soc_llt AS a
    inner join dim_soc_vigimed AS b 
        on a.LLT_NAME = b.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
        and a.PT_NAME = b.PT
        and a.HLT_NAME = b.HLT
        and a.HLGT_NAME = b.HLGT
        and a.SOC_NAME = b.SOC_VALOR
    where a.PRIMARY_FLAG = 'Y'
)

select LLT_CODE, COUNT(1) AS QT
from base
group by 1
HAVING COUNT(1) > 1
ORDER BY 2 DESC
LIMIT 10

Running query in 'duckdb:///:memory:'

,LLT_CODE,QT


In [43]:
%%sql 
SELECT distinct 
        A.LLT_NAME,
        A.LLT_CODE,
        a.PT_NAME,
        a.HLT_NAME,
        a.HLGT_NAME,
        a.SOC_NAME,
        a.PRIMARY_FLAG,
        B.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR AS "LLT",
        B.SOC_VALOR AS "SOC"
    from dim_soc_llt AS a
    inner join dim_soc_vigimed AS b 
        on a.LLT_NAME = b.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
        and a.PT_NAME = b.PT
where LLT_CODE='10050104'

Running query in 'duckdb:///:memory:'

,LLT_NAME,LLT_CODE,PT_NAME,HLT_NAME,HLGT_NAME,SOC_NAME,PRIMARY_FLAG,LLT,SOC
0,Urticária no local de aplicação,10050104,Urticária no local de aplicação,Urticárias,Angioedema e urticária,Distúrbios dos tecidos cutâneos e subcutâneos,N,Urticária no local de aplicação,Distúrbios gerais e quadros clínicos no local ...
1,Urticária no local de aplicação,10050104,Urticária no local de aplicação,Urticárias,Quadros clínicos alérgicos,Distúrbios do sistema imunitário,N,Urticária no local de aplicação,Distúrbios gerais e quadros clínicos no local ...
2,Urticária no local de aplicação,10050104,Urticária no local de aplicação,Reações no local de aplicação,Reações no local de administração,Distúrbios gerais e quadros clínicos no local ...,Y,Urticária no local de aplicação,Distúrbios gerais e quadros clínicos no local ...
3,Urticária no local de aplicação,10050104,Urticária no local de aplicação,Reações no local de aplicação,Reações no local de administração,"Lesões, intoxicações e complicações de procedi...",N,Urticária no local de aplicação,Distúrbios gerais e quadros clínicos no local ...


In [ ]:
%%sql 
SELECT  distinct 
        A.LLT_NAME,
        A.LLT_CODE,
        a.PT_NAME,
        a.HLT_NAME,
        a.HLGT_NAME,
        a.SOC_NAME,
        a.PRIMARY_FLAG,
        B.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR AS "LLT",
        B.SOC_VALOR AS "SOC"
    from dim_soc_llt AS a
    inner join dim_soc_vigimed AS b 
        on a.LLT_NAME = b.REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
        and a.PT_NAME = b.PT
where LLT_CODE='10046860'
order by 1

Running query in 'duckdb:///:memory:'

,LLT_NAME,LLT_CODE,PT_NAME,HLT_NAME,HLGT_NAME,SOC_NAME,PRIMARY_FLAG,LLT,SOC
0,Reação adversa a vacinação,10046860,Reação a imunização,Complicações relacionadas a vacinação,Lesões e complicações relacionadas a procedime...,"Lesões, intoxicações e complicações de procedi...",Y,Reação adversa a vacinação,Distúrbios do sistema imunitário
1,Reação adversa a vacinação,10046860,Reação a imunização,Quadros clínicos imunitários e associados NCO,Distúrbios imunitários NCO,Distúrbios do sistema imunitário,N,Reação adversa a vacinação,Distúrbios do sistema imunitário
2,Reação adversa a vacinação,10046860,Reação a imunização,Complicações relacionadas a vacinação,Lesões e complicações relacionadas a procedime...,"Lesões, intoxicações e complicações de procedi...",Y,Reação adversa a vacinação,"Lesões, intoxicações e complicações de procedi..."
3,Reação adversa a vacinação,10046860,Reação a imunização,Quadros clínicos imunitários e associados NCO,Distúrbios imunitários NCO,Distúrbios do sistema imunitário,N,Reação adversa a vacinação,"Lesões, intoxicações e complicações de procedi..."


# dEV

In [58]:
%run ../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings
project_root = get_project_root() 

In [62]:
path = Path(project_root) / "data/01_bronze/anvisa/Reacoes/Reacoes.parquet"
bronze = pd.read_parquet(path) 
silver = (
    bronze.rename(columns={"REACAO_EVTO_ADVERSO_MEDDRA_LLT": "LLT"})
    [["SOC","HLGT","HLT", "PT","LLT"]]
    .value_counts()
    .reset_index(name='count')  # Converte para DataFrame
    .sort_values(by=["SOC","HLGT","HLT", "PT","LLT"])
)
silver.head()

,SOC,HLGT,HLT,PT,LLT,count
5507,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação,9
10322,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual,3
19042,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada,1
6446,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação,6
13021,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato,1


In [72]:
dim_soc = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet")
dim_soc.head(2)

,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME
0,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002043,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002043,Anemia por deficiência de folato
1,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002044,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002044,Anemia por deficiência de ácido fólico


In [75]:
def join_with_meddra_priority(silver_df, dim_df):
    """
    Faz join entre silver e dim_soc_llt priorizando PRIMARY_FLAG = 'Y'
    quando houver múltiplas correspondências.
    """
    # Renomear colunas para o join
    silver_renamed = silver_df.rename(columns={
        'REACAO_EVTO_ADVERSO_MEDDRA_LLT': 'LLT',
        'SOC': 'SOC',
        'HLGT': 'HLGT', 
        'HLT': 'HLT',
        'PT': 'PT'
    })
    
    dim_renamed = dim_df.rename(columns={
        'LLT_NAME': 'LLT',
        'PT_NAME': 'PT',
        'HLT_NAME': 'HLT',
        'HLGT_NAME': 'HLGT',
        'SOC_NAME': 'SOC'
    })
    
    # Fazer o join
    joined = silver_renamed.merge(
        dim_renamed,
        on=['LLT', 'PT', 'HLT', 'HLGT', 'SOC'],
        how='left'
    )
    
    # Ordenar para priorizar PRIMARY_FLAG = 'Y' (Y vem antes de N alfabeticamente invertido)
    joined['_priority'] = joined['PRIMARY_FLAG'].apply(lambda x: 0 if x == 'Y' else 1)
    
    # Remover duplicatas mantendo PRIMARY_FLAG = 'Y'
    result = (
        joined
        .sort_values('_priority')
        .drop_duplicates(subset=['LLT', 'PT', 'HLT', 'HLGT', 'SOC'], keep='first')
        .drop(columns=['_priority'])
        .reset_index(drop=True)
    )
    
    return result


In [76]:

# Uso:
result = join_with_meddra_priority(silver, dim_soc)

In [81]:
silver.shape

(19056, 6)

In [77]:
result.shape

(19056, 15)

In [79]:
result.REACAO_CHAVE.nunique()

18450

In [78]:
result.head()

,SOC,HLGT,HLT,PT,LLT,count,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLT_CODE,PT_CODE,LLT_CODE
0,"Quadros clínicos na gravidez, no puerpério e perinatais",Quadros clínicos neonatais e perinatais,Quadros clínicos relacionados à idade gestacional e ao peso,Bebê prematuro,Nascimento prematuro,52,SOC_10036585_HLGT_10028920_HLT_10018208_PT_10036590_LLT_10036594,Y,10036585,Preg,10036585,10028920,10018208,10036590,10036594
1,"Quadros clínicos na gravidez, no puerpério e perinatais",Quadros clínicos neonatais e perinatais,Quadros clínicos relacionados à idade gestacional e ao peso,Baixo peso ao nascimento,Peso baixo ao nascimento,1,SOC_10036585_HLGT_10028920_HLT_10018208_PT_10067508_LLT_10004961,Y,10036585,Preg,10036585,10028920,10018208,10067508,10004961
2,"Quadros clínicos na gravidez, no puerpério e perinatais",Quadros clínicos neonatais e perinatais,Quadros clínicos relacionados à idade gestacional e ao peso,Baixo peso ao nascimento,Peso muito baixo do bebê ao nascimento,1,SOC_10036585_HLGT_10028920_HLT_10018208_PT_10067508_LLT_10067509,Y,10036585,Preg,10036585,10028920,10018208,10067508,10067509
3,"Quadros clínicos na gravidez, no puerpério e perinatais",Quadros clínicos neonatais e perinatais,Quadros clínicos relacionados à idade gestacional e ao peso,Bebê grande em relação à idade gestacional,Bebê grande em relação à idade gestacional,2,SOC_10036585_HLGT_10028920_HLT_10018208_PT_10023789_LLT_10023789,Y,10036585,Preg,10036585,10028920,10018208,10023789,10023789
4,"Quadros clínicos na gravidez, no puerpério e perinatais",Quadros clínicos neonatais e perinatais,Quadros clínicos relacionados à idade gestacional e ao peso,Bebê grande em relação à idade gestacional,Tamanho grande em relação à idade gestacional,1,SOC_10036585_HLGT_10028920_HLT_10018208_PT_10023789_LLT_10023790,Y,10036585,Preg,10036585,10028920,10018208,10023789,10023790


In [83]:
import unicodedata
import re

def normalize_text(text):
    """Normaliza texto para comparação: remove acentos, caracteres especiais, padroniza espaços."""
    if pd.isna(text):
        return ''
    text = str(text).lower().strip()
    # Remove acentos
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    # Remove caracteres especiais (mantém apenas letras, números e espaços)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Remove espaços múltiplos
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_columns(df, columns):
    """Cria colunas normalizadas para join."""
    df = df.copy()
    for col in columns:
        df[f'{col}_NORM'] = df[col].apply(normalize_text)
    return df


# Colunas para normalizar
join_cols = ['LLT', 'PT', 'HLT', 'HLGT', 'SOC']
norm_cols = [f'{col}_NORM' for col in join_cols]

# Preparar silver
silver_prep = silver.rename(columns={'REACAO_EVTO_ADVERSO_MEDDRA_LLT': 'LLT'})
silver_prep = normalize_columns(silver_prep, join_cols)

# Preparar dim (renomear colunas para compatibilidade)
dim_prep = dim_soc.rename(columns={
    'LLT_NAME': 'LLT',
    'PT_NAME': 'PT',
    'HLT_NAME': 'HLT',
    'HLGT_NAME': 'HLGT',
    'SOC_NAME': 'SOC'
})
dim_prep = normalize_columns(dim_prep, join_cols)

# Priorizar PRIMARY_FLAG = 'Y' (ordenar para que Y venha primeiro)
dim_prep = dim_prep.sort_values('PRIMARY_FLAG', ascending=False)

# Remover duplicatas mantendo apenas a primeira (PRIMARY_FLAG = 'Y')
dim_dedup = dim_prep.drop_duplicates(subset=norm_cols, keep='first')

# Fazer o join usando colunas normalizadas
joined = silver_prep.merge(
    dim_dedup,
    on=norm_cols,
    how='left',
    suffixes=('_VIGIMED', '_MEDDRA')
)

# Verificar resultado
print(f"Silver: {len(silver_prep)} linhas")
print(f"Matched: {joined['LLT_MEDDRA'].notna().sum()} linhas")
print(f"Não matched: {joined['LLT_MEDDRA'].isna().sum()} linhas")

Silver: 19056 linhas
Matched: 18454 linhas
Não matched: 602 linhas


In [85]:
joined.head()

,SOC_VIGIMED,HLGT_VIGIMED,HLT_VIGIMED,PT_VIGIMED,LLT_VIGIMED,count,LLT_NORM,PT_NORM,HLT_NORM,HLGT_NORM,SOC_NORM,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_MEDDRA,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_MEDDRA,HLT_CODE,HLT_MEDDRA,PT_CODE,PT_MEDDRA,LLT_CODE,LLT_MEDDRA
0,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação,9,amamentacao,amamentacao,circunstancias relacionadas a gravidez,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10036569_PT_10006247_LLT_10006247,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10036569,Circunstâncias relacionadas à gravidez,10006247,Amamentação,10006247,Amamentação
1,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual,3,abstinencia sexual,abstinencia sexual,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10086592_LLT_10086592,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10086592,Abstinência sexual,10086592,Abstinência sexual
2,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada,1,atividade sexual aumentada,atividade sexual aumentada,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10055004_LLT_10055004,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10055004,Atividade sexual aumentada,10055004,Atividade sexual aumentada
3,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação,6,nao consumacao,nao consumacao,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10029540_LLT_10029540,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10029540,Não consumação,10029540,Não consumação
4,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato,1,neonato,bebe,questoes relacionadas a idade,fatores relacionados a idade,circunstancias sociais,SOC_10041244_HLGT_10001474_HLT_10001475_PT_10021731_LLT_10028977,Y,10041244,Circunstâncias sociais,SocCi,10041244,10001474,Fatores relacionados à idade,10001475,Questões relacionadas à idade,10021731,Bebê,10028977,Neonato


In [89]:
# Ver quantidade de registros sem match
print(f"Total sem match: {joined['LLT_MEDDRA'].isna().sum()}")

Total sem match: 602


In [ ]:
no_match = joined[joined['LLT_MEDDRA'].isna()]
no_match.head(10)

,SOC_VIGIMED,HLGT_VIGIMED,HLT_VIGIMED,PT_VIGIMED,LLT_VIGIMED,count,LLT_NORM,PT_NORM,HLT_NORM,HLGT_NORM,SOC_NORM,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_MEDDRA,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_MEDDRA,HLT_CODE,HLT_MEDDRA,PT_CODE,PT_MEDDRA,LLT_CODE,LLT_MEDDRA
100,Circunstâncias sociais,Questões econômicas e habitacionais,Circunstâncias econômicas,Problema econômico,Problema econômico,5,problema economico,problema economico,circunstancias economicas,questoes economicas e habitacionais,circunstancias sociais,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101,Circunstâncias sociais,Questões econômicas e habitacionais,Circunstâncias econômicas,Problema econômico,Problema monetário SOE,1,problema monetario soe,problema economico,circunstancias economicas,questoes economicas e habitacionais,circunstancias sociais,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
401,Distúrbios cardíacos,Distúrbios do miocárdio,Cardiomiopatias,Cardiomiopatia congestiva,Cardiomiopatia congestiva,4,cardiomiopatia congestiva,cardiomiopatia congestiva,cardiomiopatias,disturbios do miocardio,disturbios cardiacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
402,Distúrbios cardíacos,Distúrbios do miocárdio,Cardiomiopatias,Cardiomiopatia congestiva,Cardiomiopatia dilatada,12,cardiomiopatia dilatada,cardiomiopatia congestiva,cardiomiopatias,disturbios do miocardio,disturbios cardiacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
468,Distúrbios cardíacos,Distúrbios do miocárdio,Miocardite não infecciosa,Miocardite,Miopericardite,14,miopericardite,miocardite,miocardite nao infecciosa,disturbios do miocardio,disturbios cardiacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
469,Distúrbios cardíacos,Distúrbios do miocárdio,Miocardite não infecciosa,Miocardite,Perimiocardite,3,perimiocardite,miocardite,miocardite nao infecciosa,disturbios do miocardio,disturbios cardiacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
525,Distúrbios cardíacos,"Distúrbios, sinais e sintomas cardíacos NCO",Distúrbios cardíacos NCO,Trombo intracardíaco,Trombo intracardíaco,9,trombo intracardiaco,trombo intracardiaco,disturbios cardiacos nco,disturbios sinais e sintomas cardiacos nco,disturbios cardiacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
526,Distúrbios cardíacos,"Distúrbios, sinais e sintomas cardíacos NCO",Distúrbios cardíacos NCO,Trombo intracardíaco,Trombose cardíaca,11,trombose cardiaca,trombo intracardiaco,disturbios cardiacos nco,disturbios sinais e sintomas cardiacos nco,disturbios cardiacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
556,Distúrbios cardíacos,Insuficiências cardíacas,Insuficiências cardíacas NCO,Insuficiência cardíaca,Insuficiência cardíaca com fração de ejeção reduzida,1,insuficiencia cardiaca com fracao de ejecao reduzida,insuficiencia cardiaca,insuficiencias cardiacas nco,insuficiencias cardiacas,disturbios cardiacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
621,"Distúrbios congênitos, de família e genéticos",Distúrbios cardíacos e vasculares congênitos,Defeitos do septo cardíaco congênitos,Cardiomiopatia hipertrófica,Cardiomiopatia hipertrófica,1,cardiomiopatia hipertrofica,cardiomiopatia hipertrofica,defeitos do septo cardiaco congenitos,disturbios cardiacos e vasculares congenitos,disturbios congenitos de familia e geneticos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
miopericardite	

In [94]:
%%sql 
SELECT LLT_NAME,PT_NAME,HLT_NAME,HLGT_NAME,SOC_NAME,PRIMARY_FLAG
from dim_soc_llt
where lower(LLT_NAME) LIKE '%miopericardite%'

Running query in 'duckdb:///:memory:'

,LLT_NAME,PT_NAME,HLT_NAME,HLGT_NAME,SOC_NAME,PRIMARY_FLAG
0,Miopericardite,Miopericardite,Miocardite não infecciosa,Distúrbios do miocárdio,Distúrbios cardíacos,Y
1,Miopericardite por Coxsackie,Miocardite por Coxsackie,"Miocardite, infecciosa",Distúrbios do miocárdio,Distúrbios cardíacos,N
2,Miopericardite por Coxsackie,Miocardite por Coxsackie,Infecções por vírus Coxsackie,Distúrbios infecciosos virais,Infecções e infestações,Y


In [96]:
%%sql 
SELECT REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR AS LLT 
,PT
,HLT
,HLGT
,SOC_VALOR AS SOC
from dim_soc_vigimed
where lower(REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR) LIKE '%miopericardite%'

Running query in 'duckdb:///:memory:'

,LLT,PT,HLT,HLGT,SOC
0,Miopericardite,Miocardite,Miocardite não infecciosa,Distúrbios do miocárdio,Distúrbios cardíacos
1,Miopericardite,Miopericardite,Miocardite não infecciosa,Distúrbios do miocárdio,Distúrbios cardíacos


In [98]:
from rapidfuzz import fuzz, process
from rapidfuzz.distance import Levenshtein
import numpy as np

def fuzzy_match_meddra_fast(no_match_df, dim_df, threshold=80):
    """
    Versão otimizada usando apenas LLT para busca inicial,
    depois valida com as outras colunas.
    """
    
    # Criar lookup de LLT no MedDRA
    dim_llt_list = dim_df['LLT_NORM'].dropna().unique().tolist()
    
    results = []
    
    for idx, row in no_match_df.iterrows():
        llt_vigimed = str(row['LLT_NORM']) if pd.notna(row['LLT_NORM']) else ''
        
        if not llt_vigimed:
            results.append({**row.to_dict(), 'MATCH_SCORE': 0})
            continue
        
        # Buscar top 5 matches por LLT
        matches = process.extract(
            llt_vigimed, 
            dim_llt_list, 
            scorer=fuzz.ratio, 
            limit=5
        )
        
        best_score = 0
        best_match = None
        
        for match_llt, llt_score, _ in matches:
            # Filtrar dim_df por esse LLT
            candidates = dim_df[dim_df['LLT_NORM'] == match_llt]
            
            for _, dim_row in candidates.iterrows():
                # Calcular score combinado
                scores = {
                    'LLT': llt_score,
                    'PT': fuzz.ratio(str(row.get('PT_NORM', '')), str(dim_row.get('PT_NORM', ''))),
                    'HLT': fuzz.ratio(str(row.get('HLT_NORM', '')), str(dim_row.get('HLT_NORM', ''))),
                    'HLGT': fuzz.ratio(str(row.get('HLGT_NORM', '')), str(dim_row.get('HLGT_NORM', ''))),
                    'SOC': fuzz.ratio(str(row.get('SOC_NORM', '')), str(dim_row.get('SOC_NORM', '')))
                }
                
                # Score ponderado
                combined_score = (
                    scores['LLT'] * 0.4 +
                    scores['PT'] * 0.25 +
                    scores['HLT'] * 0.15 +
                    scores['HLGT'] * 0.1 +
                    scores['SOC'] * 0.1
                )
                
                # Bônus para PRIMARY_FLAG
                if dim_row.get('PRIMARY_FLAG') == 'Y':
                    combined_score += 1
                
                if combined_score > best_score:
                    best_score = combined_score
                    best_match = dim_row
        
        result = row.to_dict()
        result['MATCH_SCORE'] = round(best_score, 2)
        
        if best_score >= threshold and best_match is not None:
            result['LLT_CODE'] = best_match['LLT_CODE']
            result['LLT_MEDDRA'] = best_match['LLT_NAME']
            result['PT_CODE'] = best_match['PT_CODE']
            result['PT_MEDDRA'] = best_match['PT_NAME']
            result['HLT_CODE'] = best_match['HLT_CODE']
            result['HLT_MEDDRA'] = best_match['HLT_NAME']
            result['HLGT_CODE'] = best_match['HLGT_CODE']
            result['HLGT_MEDDRA'] = best_match['HLGT_NAME']
            result['SOC_CODE'] = best_match['SOC_CODE']
            result['SOC_MEDDRA'] = best_match['SOC_NAME']
            result['PRIMARY_FLAG'] = best_match['PRIMARY_FLAG']
        
        results.append(result)
    
    return pd.DataFrame(results)



In [101]:
import unicodedata
import re

def normalize_text(text):
    """Remove acentos, converte para minúsculas e remove caracteres especiais."""
    if pd.isna(text):
        return None
    text = str(text).lower()
    # Remove acentos
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    # Remove caracteres especiais (mantém apenas letras, números e espaços)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Remove espaços múltiplos
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Criar colunas normalizadas no dim_soc
dim_soc['LLT_NORM'] = dim_soc['LLT_NAME'].apply(normalize_text)
dim_soc['PT_NORM'] = dim_soc['PT_NAME'].apply(normalize_text)
dim_soc['HLT_NORM'] = dim_soc['HLT_NAME'].apply(normalize_text)
dim_soc['HLGT_NORM'] = dim_soc['HLGT_NAME'].apply(normalize_text)
dim_soc['SOC_NORM'] = dim_soc['SOC_NAME'].apply(normalize_text)

In [102]:
# Uso:
fuzzy_results = fuzzy_match_meddra_fast(no_match, dim_soc, threshold=75)

In [103]:


# Ver resultados
fuzzy_results[fuzzy_results['MATCH_SCORE'] >= 75][
    ['LLT_VIGIMED', 'LLT_MEDDRA', 'MATCH_SCORE']
].head(20)

,LLT_VIGIMED,LLT_MEDDRA,MATCH_SCORE
0,Problema econômico,Problema econômico,90.44
1,Problema monetário SOE,Problema monetário SOE,90.44
2,Cardiomiopatia congestiva,Cardiomiopatia congestiva,93.71
3,Cardiomiopatia dilatada,Cardiomiopatia dilatada,93.71
4,Miopericardite,Miopericardite,96.83
5,Perimiocardite,Perimiocardite,96.83
6,Trombo intracardíaco,Trombo intracardíaco,90.29
7,Trombose cardíaca,Trombose cardíaca,90.29
8,Insuficiência cardíaca com fração de ejeção reduzida,Insuficiência cardíaca com fração de ejeção reduzida,85.10
9,Cardiomiopatia hipertrófica,Cardiomiopatia hipertrófica,96.42


# unindo funcoes

In [2]:
%run ../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings
project_root = get_project_root() 

In [4]:
from utils import meddra_match_pipeline, get_match_summary

In [7]:
 

meddra = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet") 
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Reacoes/Reacoes.parquet")
silver = (
    bronze.rename(columns={"REACAO_EVTO_ADVERSO_MEDDRA_LLT": "LLT"})
    [["SOC","HLGT","HLT", "PT","LLT"]]
    .value_counts()
    .reset_index(name='count')  # Converte para DataFrame
    .sort_values(by=["SOC","HLGT","HLT", "PT","LLT"])
)
silver.head()

,SOC,HLGT,HLT,PT,LLT,count
5507,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação,9
10322,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual,3
19042,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada,1
6446,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação,6
13021,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato,1


In [10]:
silver.columns

Index(['SOC', 'HLGT', 'HLT', 'PT', 'LLT', 'count'], dtype='object')

In [ ]:
result = meddra_match_pipeline(
    silver,
    meddra,
    fuzzy_threshold=60,
    output_mode='minimal',
    key_column='REACAO_CHAVE',
    extra_columns=['SOC', 'HLGT', 'HLT', 'PT', 'LLT']
) 

Match exato: 18454 registros
Sem match: 602 registros
Executando fuzzy matching com threshold=60...
Fuzzy matched: 602 registros
Sem match final: 0 registros


In [23]:
# Guardar colunas originais antes do match
original_cols = silver.columns.tolist()

result = meddra_match_pipeline(
    silver,
    meddra,
    fuzzy_threshold=60,
    output_mode='all',  # usar 'all' temporariamente
)
result.head()

Match exato: 18454 registros
Sem match: 602 registros
Executando fuzzy matching com threshold=60...
Fuzzy matched: 602 registros
Sem match final: 0 registros


,SOC_VIGIMED,HLGT_VIGIMED,HLT_VIGIMED,PT_VIGIMED,LLT_VIGIMED,count,LLT_NORM,PT_NORM,HLT_NORM,HLGT_NORM,SOC_NORM,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_MEDDRA,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_MEDDRA,HLT_CODE,HLT_MEDDRA,PT_CODE,PT_MEDDRA,LLT_CODE,LLT_MEDDRA,MATCH_TYPE,MATCH_SCORE
0,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação,9,amamentacao,amamentacao,circunstancias relacionadas a gravidez,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10036569_PT_10006247_LLT_10006247,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10036569,Circunstâncias relacionadas à gravidez,10006247,Amamentação,10006247,Amamentação,EXACT,100.0
1,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual,3,abstinencia sexual,abstinencia sexual,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10086592_LLT_10086592,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10086592,Abstinência sexual,10086592,Abstinência sexual,EXACT,100.0
2,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada,1,atividade sexual aumentada,atividade sexual aumentada,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10055004_LLT_10055004,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10055004,Atividade sexual aumentada,10055004,Atividade sexual aumentada,EXACT,100.0
3,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação,6,nao consumacao,nao consumacao,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10029540_LLT_10029540,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10029540,Não consumação,10029540,Não consumação,EXACT,100.0
4,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato,1,neonato,bebe,questoes relacionadas a idade,fatores relacionados a idade,circunstancias sociais,SOC_10041244_HLGT_10001474_HLT_10001475_PT_10021731_LLT_10028977,Y,10041244,Circunstâncias sociais,SocCi,10041244,10001474,Fatores relacionados à idade,10001475,Questões relacionadas à idade,10021731,Bebê,10028977,Neonato,EXACT,100.0


In [24]:
result.head()

,SOC_VIGIMED,HLGT_VIGIMED,HLT_VIGIMED,PT_VIGIMED,LLT_VIGIMED,count,LLT_NORM,PT_NORM,HLT_NORM,HLGT_NORM,SOC_NORM,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_MEDDRA,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_MEDDRA,HLT_CODE,HLT_MEDDRA,PT_CODE,PT_MEDDRA,LLT_CODE,LLT_MEDDRA,MATCH_TYPE,MATCH_SCORE
0,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação,9,amamentacao,amamentacao,circunstancias relacionadas a gravidez,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10036569_PT_10006247_LLT_10006247,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10036569,Circunstâncias relacionadas à gravidez,10006247,Amamentação,10006247,Amamentação,EXACT,100.0
1,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual,3,abstinencia sexual,abstinencia sexual,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10086592_LLT_10086592,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10086592,Abstinência sexual,10086592,Abstinência sexual,EXACT,100.0
2,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada,1,atividade sexual aumentada,atividade sexual aumentada,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10055004_LLT_10055004,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10055004,Atividade sexual aumentada,10055004,Atividade sexual aumentada,EXACT,100.0
3,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação,6,nao consumacao,nao consumacao,questoes de sexualidade,fatores relacionados ao genero,circunstancias sociais,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10029540_LLT_10029540,Y,10041244,Circunstâncias sociais,SocCi,10041244,10018057,Fatores relacionados ao gênero,10040489,Questões de sexualidade,10029540,Não consumação,10029540,Não consumação,EXACT,100.0
4,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato,1,neonato,bebe,questoes relacionadas a idade,fatores relacionados a idade,circunstancias sociais,SOC_10041244_HLGT_10001474_HLT_10001475_PT_10021731_LLT_10028977,Y,10041244,Circunstâncias sociais,SocCi,10041244,10001474,Fatores relacionados à idade,10001475,Questões relacionadas à idade,10021731,Bebê,10028977,Neonato,EXACT,100.0


In [19]:
result.head()

,count,REACAO_CHAVE,MATCH_TYPE,MATCH_SCORE
0,9,SOC_10041244_HLGT_10018057_HLT_10036569_PT_10006247_LLT_10006247,EXACT,100.0
1,3,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10086592_LLT_10086592,EXACT,100.0
2,1,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10055004_LLT_10055004,EXACT,100.0
3,6,SOC_10041244_HLGT_10018057_HLT_10040489_PT_10029540_LLT_10029540,EXACT,100.0
4,1,SOC_10041244_HLGT_10001474_HLT_10001475_PT_10021731_LLT_10028977,EXACT,100.0


In [ ]:
from utils import meddra_match_pipeline

# Modo 1: Todas as colunas (comportamento original)
result = meddra_match_pipeline(
    silver,
    meddra_df,
    fuzzy_threshold=60,
    output_mode='all'  # default
)

# Modo 2: Colunas originais + REACAO_CHAVE + MATCH_TYPE + MATCH_SCORE
result = meddra_match_pipeline(
    silver,
    meddra_df,
    fuzzy_threshold=60,
    output_mode='slim',
    key_column='REACAO_CHAVE'
)

# Modo 3: Apenas colunas originais + REACAO_CHAVE (sem metadados de match)
result = meddra_match_pipeline(
    silver,
    meddra_df,
    fuzzy_threshold=60,
    output_mode='minimal',
    key_column='REACAO_CHAVE'
)

# Com colunas extras do MedDRA
result = meddra_match_pipeline(
    silver,
    meddra_df,
    fuzzy_threshold=60,
    output_mode='slim',
    key_column='REACAO_CHAVE',
    extra_columns=['LLT_CODE', 'PT_CODE', 'PRIMARY_FLAG']
)

In [8]:


# Carregar dados
vigimed_df = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Reacoes/Reacoes.parquet")
meddra_df = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet")

# Executar pipeline completo (exato + fuzzy)
result = meddra_match_pipeline(
    silver, #vigimed_df,
    meddra_df,
    fuzzy_threshold=60,
)

# Ver resumo
get_match_summary(result)

Match exato: 18454 registros
Sem match: 602 registros
Executando fuzzy matching com threshold=60...
Fuzzy matched: 602 registros
Sem match final: 0 registros


,MATCH_TYPE,count,avg_score,min_score,max_score,pct
0,EXACT,18454,100.00000,100.00,100.0,96.84
1,FUZZY,602,93.41397,60.22,100.4,3.16


In [10]:
result.query("MATCH_TYPE == 'NO_MATCH'")

,SOC_VIGIMED,HLGT_VIGIMED,HLT_VIGIMED,PT_VIGIMED,LLT_VIGIMED,count,LLT_NORM,PT_NORM,HLT_NORM,HLGT_NORM,SOC_NORM,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_MEDDRA,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_MEDDRA,HLT_CODE,HLT_MEDDRA,PT_CODE,PT_MEDDRA,LLT_CODE,LLT_MEDDRA,MATCH_TYPE,MATCH_SCORE
